# Double Overtime?

## I. Summary
Here, we search for officers who engage in two overtime shifts simultaneously;  
We find that there were likely no such instances.

## II. Importing libraries and datasets

In [1]:
import pandas as pd
from datetime import datetime

In [2]:
df_court = pd.read_csv("../cleaned_data/Court_merged_cleaned.csv")
df_detail = pd.read_csv("../cleaned_data/Detail_merged_cleaned.csv")
df_events = pd.read_csv("../cleaned_data/Events_merged_cleaned.csv")
df_records = pd.read_csv("../cleaned_data/Records_merged_cleaned.csv")

## III. Formatting datasets

In [3]:
df_court.columns, df_detail.columns, df_events.columns, df_records.columns

(Index(['IDNO6', 'NAME', 'RANK', 'ASSIGNED_DESC', 'CHARGED_DESC', 'OTDATE',
        'STARTTIME', 'ENDTIME', 'WRKDHRS'],
       dtype='object'),
 Index(['RowId', 'Tracking_No', 'Emp_No', 'Emp_Name', 'Emp_Rank',
        'Emp_Org_Code', 'District_Worked', 'Customer_No', 'Customer_Seq',
        'Customer_No_And_Seq', 'BPD_Customer_no', 'Street', 'Location',
        'Start_date', 'Start_Time', 'End_Time', 'Hours_Worked', 'Detail_Type',
        'Detail_Rank', 'Admin_Fee_Flag', 'Pay_Hours', 'Pay_Amount',
        'Pay_TRC_Code', 'Pay_Rate', 'FBK_Pay_Date'],
       dtype='object'),
 Index(['ID', 'NAME', 'RANK', 'ASSIGNED_DESC', 'CHARGED_DESC', 'OTDATE',
        'DESCRIPTION', 'STARTTIME', 'ENDTIME', 'OTHOURS'],
       dtype='object'),
 Index(['IDNO6', 'NAME', 'RANK', 'ASSIGNED_DESC', 'CHARGED_DESC', 'OTDATE',
        'DESCRIPTION', 'STARTTIME', 'ENDTIME', 'WRKDHRS', 'OTHOURS'],
       dtype='object'))

In [4]:
df_court['OTDATE'] = pd.to_datetime(df_court['OTDATE']).dt.strftime('%Y-%m-%d')
df_events['OTDATE'] = pd.to_datetime(df_events['OTDATE']).dt.strftime('%Y-%m-%d')

C:\Users\taiyo\AppData\Local\Temp\ipykernel_27028\2716188621.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_events['OTDATE'] = pd.to_datetime(df_events['OTDATE']).dt.strftime('%Y-%m-%d')


In [25]:
df_court1 = df_court.rename(columns={'WRKDHRS': 'OTHOURS'})
df_events1 = df_events.drop(['DESCRIPTION'], axis=1).rename(columns={'ID': 'IDNO6'})
df_records1 = df_records.drop(['DESCRIPTION', 'WRKDHRS'], axis=1)
#df_combined = pd.concat([df_court1, df_events1, df_records1])

In [26]:
df_court1.shape, df_events1.shape, df_records1.shape

((39750, 9), (42196, 9), (406022, 9))

In [ ]:
#df_sameday = df_combined[df_combined.duplicated(subset=['IDNO6', 'OTDATE'], keep=False)].sort_values(['IDNO6', 'OTDATE', 'STARTTIME'])

## IV. Searching for double dipping

In [60]:
def check_overlap(start1, start2, end1, end2):
    return (start1 < end2 and end1 > start2) #or (start1 <= end2 and end1 > start2)

In [31]:
df_court1.dtypes, df_events1.dtypes, df_records1.dtypes

(IDNO6            float64
 NAME              object
 RANK              object
 ASSIGNED_DESC     object
 CHARGED_DESC      object
 OTDATE            object
 STARTTIME        float64
 ENDTIME          float64
 OTHOURS          float64
 dtype: object,
 IDNO6            float64
 NAME              object
 RANK              object
 ASSIGNED_DESC     object
 CHARGED_DESC      object
 OTDATE            object
 STARTTIME        float64
 ENDTIME          float64
 OTHOURS          float64
 dtype: object,
 IDNO6            float64
 NAME              object
 RANK              object
 ASSIGNED_DESC     object
 CHARGED_DESC      object
 OTDATE            object
 STARTTIME        float64
 ENDTIME          float64
 OTHOURS          float64
 dtype: object)

In [30]:
df_court1['STARTTIME'] = df_court1['STARTTIME'].astype(float)
df_court1['ENDTIME'] = df_court1['ENDTIME'].astype(float)

df_records1['STARTTIME'] = df_records1['STARTTIME'].astype(float)
df_records1['ENDTIME'] = df_records1['ENDTIME'].astype(float)

### Court and Special Events
*Note: The code below is partially from generative AI*

In [61]:
# Define a function to find overlaps within each group for df_court1 and df_events1
def find_overlaps(group):
    results = []
    # Compare each row in df_court1 with rows in df_events1
    for _, row_court in group['df_court1'].iterrows():
        for _, row_event in group['df_events1'].iterrows():
            # Check for overlap between df_court1 and df_events1
            if check_overlap(row_court["STARTTIME"], row_event["STARTTIME"], row_court["ENDTIME"], row_event["ENDTIME"]):
                results.append({
                    "IDNO6": row_court["IDNO6"],
                    "OTDATE": row_court["OTDATE"],
                    "court Start": row_court["STARTTIME"],
                    "court End": row_court["ENDTIME"],
                    "event Start": row_event["STARTTIME"],
                    "event End": row_event["ENDTIME"]
                })
    return pd.DataFrame(results)

# Group each dataframe by "IDNO6" and "OTDATE" and convert to dictionary of DataFrames
court_groups = {name: group for name, group in df_court1.groupby(["IDNO6", "OTDATE"])}
event_groups = {name: group for name, group in df_events1.groupby(["IDNO6", "OTDATE"])}

# Iterate over each group key (IDNO6, OTDATE) to find overlaps
overlapping_results = []
for key in court_groups.keys():
    if key in event_groups:
        group = {
            'df_court1': court_groups[key],
            'df_events1': event_groups[key]
        }
        overlapping_results.append(find_overlaps(group))

# Combine all overlapping results into a single DataFrame
overlapping_timespans = pd.concat(overlapping_results, ignore_index=True)

In [62]:
overlapping_timespans

""


### Court, Special Events, and Records Request
Records Request contains some rows of `df_court1` and `df_events1`, so the code below may flag these as overlapping.

In [63]:
# Define a function to find overlaps within each group for at least two tables
def find_overlaps(group):
    results = []
    # Compare each row in df_court1 with rows in df_events1 and df_records1
    for _, row_court in group['df_court1'].iterrows():
        for _, row_event in group['df_events1'].iterrows():
            # Check for overlap between df_court1 and df_events1
            if check_overlap(row_court["STARTTIME"], row_event["STARTTIME"], row_court["ENDTIME"], row_event["ENDTIME"]):
                results.append({
                    "IDNO6": row_court["IDNO6"],
                    "OTDATE": row_court["OTDATE"],
                    "source": "court & events",
                    "court Start": row_court["STARTTIME"],
                    "court End": row_court["ENDTIME"],
                    "event Start": row_event["STARTTIME"],
                    "event End": row_event["ENDTIME"]
                })

        for _, row_record in group['df_records1'].iterrows():
            # Check for overlap between df_court1 and df_records1
            if check_overlap(row_court["STARTTIME"], row_record["STARTTIME"], row_court["ENDTIME"], row_record["ENDTIME"]):
                results.append({
                    "IDNO6": row_court["IDNO6"],
                    "OTDATE": row_court["OTDATE"],
                    "source": "court & records",
                    "court Start": row_court["STARTTIME"],
                    "court End": row_court["ENDTIME"],
                    "record Start": row_record["STARTTIME"],
                    "record End": row_record["ENDTIME"]
                })

        for _, row_event in group['df_events1'].iterrows():
            for _, row_record in group['df_records1'].iterrows():
                # Check for overlap between df_events1 and df_records1
                if check_overlap(row_event["STARTTIME"], row_record["STARTTIME"], row_event["ENDTIME"], row_record["ENDTIME"]):
                    results.append({
                        "IDNO6": row_event["IDNO6"],
                        "OTDATE": row_event["OTDATE"],
                        "source": "events & records",
                        "event Start": row_event["STARTTIME"],
                        "event End": row_event["ENDTIME"],
                        "record Start": row_record["STARTTIME"],
                        "record End": row_record["ENDTIME"]
                    })
    return pd.DataFrame(results)

# Group each dataframe by "IDNO6" and "OTDATE" and convert to dictionary of DataFrames
court_groups = {name: group for name, group in df_court1.groupby(["IDNO6", "OTDATE"])}
event_groups = {name: group for name, group in df_events1.groupby(["IDNO6", "OTDATE"])}
record_groups = {name: group for name, group in df_records1.groupby(["IDNO6", "OTDATE"])}

# Iterate over each group key (IDNO6, OTDATE) to find overlaps
overlapping_results = []
for key in court_groups.keys():
    if key in event_groups and key in record_groups:
        group = {
            'df_court1': court_groups[key],
            'df_events1': event_groups[key],
            'df_records1': record_groups[key]
        }
        overlapping_results.append(find_overlaps(group))

# Combine all overlapping results into a single DataFrame
overlapping_timespans = pd.concat(overlapping_results, ignore_index=True)


In [64]:
overlapping_timespans

,IDNO6,OTDATE,source,court Start,court End,record Start,record End,event Start,event End
0,9014.0,2020-02-10,court & records,830.0,1000.0,830.0,1000.0,NaN,NaN
1,10090.0,2020-01-24,court & records,830.0,900.0,830.0,900.0,NaN,NaN
2,10090.0,2020-01-24,events & records,NaN,NaN,1600.0,2345.0,1600.0,2345.0
3,10090.0,2020-03-13,court & records,1200.0,1300.0,1200.0,1300.0,NaN,NaN
4,10396.0,2020-02-07,court & records,930.0,1030.0,930.0,1030.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...
139,157686.0,2021-10-26,events & records,NaN,NaN,1600.0,2345.0,1600.0,2345.0
140,157695.0,2021-08-06,court & records,900.0,930.0,900.0,930.0,NaN,NaN
141,157695.0,2021-08-06,events & records,NaN,NaN,1600.0,2345.0,1600.0,2345.0
142,162930.0,2021-12-29,court & records,900.0,930.0,900.0,930.0,NaN,NaN


### Records Request

In [65]:
def check_overlap(start1, start2, end1, end2):
    return (start1 < end2 and end1 > start2) #or (start1 <= end2 and end1 > start2)

# Define a function to find overlaps within each group in df_records1
def find_internal_overlaps(group):
    results = []
    # Compare each row with every other row in the same group
    for i, row1 in group.iterrows():
        for j, row2 in group.iterrows():
            if i >= j:  # Avoid double-counting and self-comparison
                continue
            # Check for overlap within df_records1
            if check_overlap(row1["STARTTIME"], row2["STARTTIME"], row1["ENDTIME"], row2["ENDTIME"]):
                results.append({
                    "IDNO6": row1["IDNO6"],
                    "OTDATE": row1["OTDATE"],
                    "Start1": row1["STARTTIME"],
                    "End1": row1["ENDTIME"],
                    "Start2": row2["STARTTIME"],
                    "End2": row2["ENDTIME"]
                })
    return pd.DataFrame(results)

# Group df_records1 by "IDNO6" and "OTDATE" and check for internal overlaps in each group
overlapping_results = []
for name, group in df_records1.groupby(["IDNO6", "OTDATE"]):
    overlapping_results.append(find_internal_overlaps(group))

# Combine all overlapping results into a single DataFrame
overlapping_timespans = pd.concat(overlapping_results, ignore_index=True)


In [66]:
overlapping_timespans

""


We can infer from the above `overlapping_timespans` being empty that there were no instances of double-dipping across overtime shifts.